# Set 11 – Neuronales Netz für Klassifikation

Dieses Notebook behandelt eine Klassifikation mit **drei Klassen**. Das Netz besitzt deshalb drei Output-Neuronen. Ihre Softmax-Werte lassen sich als Klassenwahrscheinlichkeiten interpretieren und summieren sich pro Beobachtung zu 1.

## Lernziele

- `MLPClassifier` mit drei Output-Neuronen einsetzen
- Softmax und Cross-Entropy miteinander verbinden
- `predict`, `predict_proba` und `classes_` unterscheiden
- Lernkurve und Entscheidungsflächen untersuchen
- Architektur, Aktivierung, Lernrate und Regularisierung optimieren

Zum Vergleich: Bei einer binären Klassifikation verwendet scikit-learn ein logistisches Output-Neuron. Bei drei Klassen werden drei Softmax-Ausgänge verwendet.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
plt.style.use("seaborn-v0_8-whitegrid")
from sklearn.datasets import make_classification
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay


## 1. Kontrollierte Daten mit drei Klassen

Wir verwenden genau zwei Inputmerkmale, damit die Entscheidungsflächen sichtbar bleiben. `class_sep` und `flip_y` steuern Trennbarkeit und Labelrauschen.


In [ ]:
X, y = make_classification(
    n_samples=650,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_classes=3,
    n_clusters_per_class=1,
    class_sep=1.15,
    flip_y=0.04,
    random_state=RANDOM_STATE,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

plt.figure(figsize=(7, 5))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="viridis", s=28, alpha=0.8)
plt.xlabel("Merkmal 1"); plt.ylabel("Merkmal 2"); plt.title("Drei Klassen")
plt.show()


## 2. Warum Softmax und Cross-Entropy?

Für drei gegenseitig ausschließende Klassen benötigt das Modell drei Scores. **Softmax** wandelt sie in Wahrscheinlichkeiten um:

$$p_k = \frac{e^{z_k}}{\sum_j e^{z_j}}$$

Die Wahrscheinlichkeiten liegen zwischen 0 und 1 und summieren sich zu 1. Der **Cross-Entropy-Loss** bestraft das Modell besonders stark, wenn es der falschen Klasse eine hohe Wahrscheinlichkeit gibt.

scikit-learn wählt Output-Aktivierung und Log-Loss automatisch. `activation="relu"` bezieht sich deshalb nur auf die Hidden Layer.


## 3. Training und zentrale Parameter

`solver="adam"` ist der Optimierer. `learning_rate_init` steuert die erste Schrittweite. `alpha` fügt dem Loss eine L2-Strafe auf große Gewichte hinzu. `batch_size` legt fest, wie viele Beobachtungen ein Gewichtsupdate verwenden. `max_iter` begrenzt die Epochen, während Early Stopping früher abbrechen kann.


In [ ]:
classifier = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=(32, 16), # zwei Hidden Layer
        activation="relu",          # Hidden-Aktivierung, nicht Output
        solver="adam",              # adaptiver Optimierer
        learning_rate_init=0.005,    # anfängliche Schrittweite
        alpha=0.001,                 # L2-Regularisierung
        batch_size=32,
        max_iter=1000,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=35,
        random_state=RANDOM_STATE,
    )),
])

classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
y_probability = classifier.predict_proba(X_test)

print("Erste Wahrscheinlichkeiten:\n", np.round(y_probability[:4], 3))
print("Zeilensummen:", y_probability[:4].sum(axis=1))
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Macro-F1:", round(f1_score(y_test, y_pred, average="macro"), 3))


## 4. Gelernte Struktur und Trainingsverlauf

`n_outputs_` und `out_activation_` bestätigen die drei Softmax-Ausgänge. Die Gewichtsmatrix zum Output besitzt für jede vorherige Hidden-Einheit drei Gewichte.


In [ ]:
mlp = classifier.named_steps["mlp"]
print("Klassen:", mlp.classes_)
print("Output-Neuronen:", mlp.n_outputs_)
print("Output-Aktivierung:", mlp.out_activation_)
print("Epochen:", mlp.n_iter_)
for index, weights in enumerate(mlp.coefs_, 1):
    print(f"Gewichtsmatrix {index}: {weights.shape}")

plt.figure(figsize=(7, 4))
plt.plot(mlp.loss_curve_, label="Trainings-Loss")
plt.xlabel("Epoche"); plt.ylabel("Log-Loss"); plt.title("Lernkurve")
plt.legend(); plt.show()


In [ ]:
def plot_decision_regions(model, X, y, title):
    xx, yy = np.meshgrid(
        np.linspace(X[:, 0].min()-0.7, X[:, 0].max()+0.7, 280),
        np.linspace(X[:, 1].min()-0.7, X[:, 1].max()+0.7, 280),
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    prediction = model.predict(grid).reshape(xx.shape)
    plt.figure(figsize=(8, 5.5))
    plt.contourf(xx, yy, prediction, alpha=0.2, cmap="viridis")
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap="viridis", s=24, alpha=0.75)
    plt.xlabel("Merkmal 1"); plt.ylabel("Merkmal 2"); plt.title(title)
    plt.show()

plot_decision_regions(classifier, X_train, y_train, "Entscheidungsflächen des MLP")
print(classification_report(y_test, y_pred, digits=3))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues")
plt.title("Konfusionsmatrix auf Testdaten"); plt.show()


## 5. Kleine Hyperparametersuche

Wir testen nur drei überschaubare Strukturen. Neben der Netzgröße untersuchen wir Hidden-Aktivierung, Lernrate und L2-Regularisierung. Der Output bleibt automatisch Softmax, weil das Problem drei Klassen besitzt.


In [ ]:
search_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        solver="adam",
        batch_size=32,
        max_iter=900,
        early_stopping=True,
        n_iter_no_change=30,
        random_state=RANDOM_STATE,
    )),
])

param_grid = {
    "mlp__hidden_layer_sizes": [(16,), (32,), (32, 16)],
    "mlp__activation": ["relu", "tanh"],
    "mlp__learning_rate_init": [0.001, 0.01],
    "mlp__alpha": [0.0001, 0.01],
}

search = GridSearchCV(
    search_pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    return_train_score=True,
)
search.fit(X_train, y_train)
print("Beste Parameter:", search.best_params_)
print("Bester CV-Macro-F1:", round(search.best_score_, 3))


In [ ]:
results = pd.DataFrame(search.cv_results_)
display(results[["params", "mean_train_score", "mean_test_score", "std_test_score"]]
        .sort_values("mean_test_score", ascending=False).head(8).round(3))

best_classifier = search.best_estimator_
best_prediction = best_classifier.predict(X_test)
print("Test-Macro-F1:", round(f1_score(y_test, best_prediction, average="macro"), 3))
plot_decision_regions(best_classifier, X_train, y_train, "Bestes MLP aus der Suche")


## Fazit

- Drei Klassen führen zu drei Softmax-Output-Neuronen.
- Cross-Entropy passt zu Wahrscheinlichkeiten und gegenseitig ausschließenden Klassen.
- `predict_proba` liefert Wahrscheinlichkeiten, `predict` die Klasse mit dem größten Wert.
- Lernrate und Netzgröße beeinflussen Lernfähigkeit und Stabilität.
- `alpha` und Early Stopping reduzieren Overfitting.
- Der Testdatensatz wird erst nach der Hyperparameterwahl verwendet.
